# Figure 3 — Embedding Space (UMAP)

### Input file
| Variable | File | Content |
|---|---|---|
| `df` | `data/umap_coordinates.parquet` | 2D UMAP coordinates + per-spectrum metadata (100,000 spectra × 62 cols) |

Generated from the final **LCFM TS·noPA model** (checkpoint `763ad9ee`) over 100,000 spectra
(LCFM eval set). One fixed UMAP layout; every panel is a recolour and/or zoom of that same layout.

---
### Structure
- **Part 1 — Global view**: the whole layout recoloured by each metadata field (one panel per property)
- **Part 2 — Hierarchical zoom**: recursive descent into the densest sub-cluster, recoloured level by level, then the deepest region recoloured by every remaining property
- **Part 3 — Duplicate retrieval**: spectra sharing an identical peptide `sequence` highlighted to show same-peptide co-location

All panels are saved as **.svg** into `figures/3/`, in the same publication style as Figure 1.


In [ ]:
# ── Environment check — do not pip install from inside the notebook ───────────
# The figure environment is pinned in pyproject.toml under the "figures"
# dependency group, and the pins matter: figure_3's zoom-region search ranks
# candidate windows whose separation scores frequently tie, and before the
# rankings were made total orders numpy 1.26 and 2.0 broke those ties
# differently and silently selected different regions. Installing from a cell
# cannot fix a mismatch either — once numpy is imported, `pip install numpy==x`
# does not change the module already loaded in this kernel.
#
# Create the pinned environment once and register it as a kernel:
#
#     ./setup_kernel.sh          # see README -> Reproducing the figures
#
# then select the "InstaNovo-FM (figures)" kernel and re-run.
import importlib.metadata as _im
import pathlib as _pl
import re as _re
import warnings as _w

try:
    import tomllib as _tomllib
except ModuleNotFoundError:            # Python 3.10
    _tomllib = None

_root = next((p for p in (_pl.Path.cwd(), *_pl.Path.cwd().parents)
              if (p / 'pyproject.toml').is_file()), None)

if _root is None:
    _w.warn('pyproject.toml not found; skipping the environment check.')
elif _tomllib is None:
    _w.warn('tomllib needs Python 3.11+; skipping the environment check. '
            'setup_kernel.sh builds a 3.11 environment by default.')
else:
    _pins = _tomllib.loads((_root / 'pyproject.toml').read_text())
    _want = {}
    for _spec in _pins.get('dependency-groups', {}).get('figures', []):
        _m = _re.fullmatch(r'([A-Za-z0-9_.-]+)==([0-9][^\s;]*)', str(_spec).strip())
        if _m:
            _want[_m.group(1)] = _m.group(2)

    _bad = []
    for _pkg, _ver in sorted(_want.items()):
        try:
            _got = _im.version(_pkg)
        except _im.PackageNotFoundError:
            _bad.append((_pkg, _ver, 'not installed'))
            continue
        if _got != _ver:
            _bad.append((_pkg, _ver, _got))

    if _bad:
        _msg = '\n'.join(f'  {p:16s} want {w:10s} have {g}' for p, w, g in _bad)
        # numpy decides which zoom regions figure_3 selects, so treat it as fatal.
        if any(p == 'numpy' for p, _, _ in _bad):
            raise RuntimeError(
                'Environment does not match the "figures" group in pyproject.toml:\n'
                + _msg +
                '\n\nRun ./setup_kernel.sh and select the "InstaNovo-FM (figures)" kernel.')
        _w.warn('Environment differs from the "figures" group in pyproject.toml:\n' + _msg)
    else:
        print(f'Environment matches pyproject.toml [figures] ({len(_want)} pinned packages).')


In [ ]:
from pathlib import Path as SysPath

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle
import seaborn as sns
from sklearn.cluster import KMeans
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Repo root, independent of where the kernel was started ───────────────────
# Path.cwd() only worked when Jupyter happened to be launched from the repo root;
# with the notebooks in notebooks/ that silently pointed everything one level down.
# Walk up instead, anchored on files that only exist at the root.
def _find_repo_root(start=None):
    _p = (start or SysPath.cwd()).resolve()
    for _cand in (_p, *_p.parents):
        if (_cand / 'config' / 'metadata_colors.json').is_file() and (_cand / 'data').is_dir():
            return _cand
    raise RuntimeError(
        'Repo root not found: expected an ancestor holding config/metadata_colors.json '
        f'and data/. Searched upward from {_p}.')

BASE_DIR = _find_repo_root()


def set_publication_style():
    '''
    Apply consistent, publication-ready matplotlib style.
    Uses Palatino (or nearest available serif alternative) and ticks theme.
    Color palette is coherent with Nature Methods guidelines (pastel, colorblind-safe).
    '''
    sns.set_theme(style="ticks")
    plt.rcParams.update({
        # Font — Palatino with graceful fallbacks
        "font.family":      "serif",
        "font.serif":       ["Palatino", "Palatino Linotype", "TeX Gyre Pagella",
                             "Book Antiqua", "URW Palladio L", "DejaVu Serif"],
        "font.size":        14,
        "axes.titlesize":   16,
        "axes.labelsize":   15,
        "xtick.labelsize":  12,
        "ytick.labelsize":  12,
        # Axes & ticks
        "axes.linewidth":    1.5,
        "xtick.major.width": 1,
        "ytick.major.width": 1,
        "axes.spines.top":   False,
        "axes.spines.right": False,
        # Figure
        "figure.dpi":       300,
        "figure.facecolor": "white",
        "axes.facecolor":   "white",
        # Legend
        "legend.fontsize":      13,
        "legend.frameon":       False,
        "legend.columnspacing": 1.5,
        # Grid
        "axes.grid":      True,
        "grid.alpha":     0.25,
        "grid.color":     "#CCCCCC",
        "grid.linewidth": 0.5,
    })


def get_figsize(width_ratio=1, total_width_inch=14.0):
    '''Return (width, height) in inches scaled to a standard A4 column width.'''
    ratios = {1: (1, 1), 2: (2, 1), 3: (3, 1)}
    w_mult, h_mult = ratios.get(width_ratio, (1, 1))
    actual_width  = (total_width_inch / 3) * w_mult
    actual_height = actual_width / w_mult
    return (actual_width, actual_height)

FIGURES_DIR = BASE_DIR / 'figures' / '3'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    path = str(FIGURES_DIR / name) + '.svg'
    fig.savefig(path, bbox_inches='tight')
    print(f'  Saved → figures/3/{name}.svg')

def fmt_n(n):
    if n >= 1e9: return f'{n/1e9:.1f}B'
    if n >= 1e6: return f'{n/1e6:.0f}M'
    if n >= 1e3: return f'{n/1e3:.0f}K'
    return str(int(round(n)))

set_publication_style()

---

In [ ]:
# ── Data loading ──────────────────────────────────────────────────────────────
DATA_PATH = BASE_DIR / 'data' / 'umap_coordinates.parquet'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing UMAP coordinates: {DATA_PATH}')

df = pd.read_parquet(DATA_PATH)
print(f'Loaded {len(df):,} spectra × {df.shape[1]} columns')

# Light organism cleanup: merge case-variant duplicates (e.g. "Homo Sapiens" → "Homo sapiens")
_low   = df['search_organism'].astype(str).str.lower()
_canon = df.groupby(_low)['search_organism'].agg(lambda s: s.value_counts().index[0])
df['search_organism'] = _low.map(_canon)

# ── Config — matches the source scripts (make_hierarchical_umap_figure.py) ─────
X, Y           = 'umap_x', 'umap_y'
# Top-K categories shown; the rest collapse to grey. K is the number of DISTINCT
# hues the palette actually has — asking seaborn for more just cycles it, and two
# different categories would silently share a colour.
BBOX_PCT       = (1.0, 99.0)              # robust bounding-box percentiles for zoom
DESCENT_K      = 3                        # KMeans sub-clusters per level; zoom descends into densest
POINT_SIZE     = 4
ALPHA          = 0.6
_NULLISH       = {'', 'none', 'nan', 'unknown', 'null', 'na'}

# Colorblind-safe categorical palette (coherent with figure_1 aesthetic)
CAT_COLORS     = sns.color_palette('colorblind')
MAX_CATEGORIES = len(CAT_COLORS)
assert len({tuple(c) for c in CAT_COLORS}) == MAX_CATEGORIES, 'palette has repeats'
OTHER_GREY = '#DDDDDD'

# Projects need more slots than the 10-hue CVD-safe palette provides. These 12 were
# selected greedily from tab20 + colorblind + Set1 + Dark2 to maximise the smallest
# pairwise separation under normal vision and protan/deutan/tritan simulation, while
# also staying clear of OTHER_GREY (grey is reserved for "other"). Near-neutral
# candidates were excluded for the same reason.
#   worst pair dE 5.2, worst vs grey dE 5.5
# For reference the incumbent 10-colour palette has a worst pair of dE 3.2, so this
# is more distinguishable despite carrying two more categories. 12 is the practical
# ceiling: adding a 13th drops the worst pair to ~4.6 and 20 colours reach ~2.5,
# where distinct categories become genuinely indistinguishable.
# A multi-species search database ("Homo sapiens; Arabidopsis thaliana") is not a
# species; listing it beside "Homo sapiens" in a legend reads as an error. Collapse
# every such label into one explicit category.
_multi = df['search_organism'].astype(str).str.contains(';')
df['organism_group'] = np.where(_multi, 'Mixed (multi-species)', df['search_organism'].astype(str))
print(f'{int(_multi.sum()):,} spectra have a multi-species label -> "Mixed (multi-species)"')

_NULLISH_PREVIEW = {'', 'none', 'nan', 'unknown', 'null', 'na'}

# ── Colours come from metadata_colors.json, the map shared with figure_1 ──────
with open(BASE_DIR / 'config' / 'metadata_colors.json') as _cf:
    META_COLORS = json.load(_cf)
OTHER_GREY = META_COLORS['kingdom']['Other']

# The JSON is keyed on figure_1's normalised labels, the parquet holds raw values.
# These are the same mappers figure_1 uses, so one label means one colour in both.
def inst_family(s):
    s = str(s).lower()
    if 'timstof' in s or 'tims' in s:                                   return 'TimsTOF'
    if 'astral'  in s:                                                  return 'Orbitrap Astral'
    if 'elite'   in s:                                                  return 'Orbitrap Elite'
    if 'velos'   in s:                                                  return 'Orbitrap Velos'
    if 'q exactive' in s:                                               return 'Q Exactive'
    if any(k in s for k in ('fusion', 'lumos', 'eclipse', 'exploris')): return 'Orbitrap Fusion'
    return 'Orbitrap (Other)'

def det_group(s):
    s = str(s)
    if 'Orbitrap|IonTrap' in s: return 'Orbitrap+IT'
    if 'Orbitrap' in s:         return 'Orbitrap'
    if 'Astral' in s:           return 'Astral'
    if 'TOF' in s:              return 'TOF'
    if 'Triple' in s:           return 'Triple Quad'
    return 'Ion Trap'

def charge_group(z):
    try:
        zi = int(float(z))
    except (TypeError, ValueError):
        return 'Other'
    if zi <= 0:  return 'Other'        # unassigned charge is absence, not a state
    return str(zi) if zi <= 4 else '5+'

df['instrument_family'] = df['search_instrument'].map(inst_family)
df['detector_group']    = df['search_detector'].map(det_group)
df['charge_group']      = df['precursor_charge'].map(charge_group)
df['enzyme_norm']       = df['search_enzyme'].astype(str).str.title()

# PTM class straight from the parquet; chemical labelling has to be resolved,
# because modification_types leaves the label accessions as raw UniMod numbers.
df['ptm_class'] = df['modification_class'].astype(str)

# search_quant is the authoritative labelling field: 'precursor' means label-free
# quantification, 'TMT' isobaric labelling. Deriving this from UniMod accessions in
# modification_types instead finds only 113 of the 3,610 labelled spectra, because a
# fixed TMT modification is usually not recorded per spectrum. The accessions are
# still worth consulting for the handful search_quant cannot express: iTRAQ and
# SILAC have no search_quant category and would otherwise be filed as label-free.
_LABEL_ACC = {'214': 'iTRAQ4plex', '188': 'SILAC', '259': 'SILAC',
              '267': 'SILAC', '312': 'SILAC', '481': 'SILAC'}

def _acc_label(s):
    hits = {v for a, v in _LABEL_ACC.items() if f'UniMod:{a}' in str(s)}
    if not hits:      return None
    if len(hits) > 1: return 'Mixed label'
    return hits.pop()

_quant = df['search_quant'].astype(str).map({'precursor': 'Label-free', 'TMT': 'TMT'})
_resc  = df['modification_types'].map(_acc_label)
df['chem_label'] = np.where((_quant == 'Label-free') & _resc.notna(), _resc,
                            _quant.fillna('Label-free'))

_lab = df['chem_label'].value_counts()
print('  chemical labelling (from search_quant, rescued with UniMod accessions):')
for _k, _v in _lab.items():
    print(f'    {_k:14s} {_v:7,}  ({_v/len(df)*100:5.2f}%)')

# column -> block in metadata_colors.json
CAT_COLOR_BLOCK = {
    'frag_type':         'fragmentation',
    'search_acquisition':'acquisition',
    'enzyme_norm':       'enzyme',
    'instrument_family': 'instrument',
    'detector_group':    'detector',
    'charge_group':      'charge',
    'organism_group':    'organism_full',
    'search_project':    'project',
    'ptm_class':         'modification_class',
    'chem_label':        'chem_label',
}
for _c, _b in CAT_COLOR_BLOCK.items():
    _vals = set(df[_c].astype(str)) if _c in df.columns else set()
    _miss = sorted(v for v in _vals if v not in META_COLORS[_b]
                   and v.lower() not in _NULLISH_PREVIEW)
    print(f'  {_c:20s} -> {_b:14s} {len(_vals):3d} values, '
          f'{len(_miss):3d} without a colour{" " + str(_miss[:4]) if _miss else ""}')

# The recursive zoom hierarchy: (column, label), one colour layer per level
LEVELS = [
    ('frag_type',         'Fragmentation method'),
    ('search_instrument', 'Instrument'),
    ('search_organism',   'Organism'),
    ('precursor_charge',  'Precursor charge'),
]
LEVELS = [(c, t) for c, t in LEVELS if c in df.columns]

# At the deepest zoom the SAME region is recoloured by every remaining property
BOTTOM_PROPERTIES = [
    ('precursor_mz',        'continuous',  'magma'),
    ('n_peaks',             'continuous',  'viridis'),
    ('sequence_length',     'continuous',  'cividis'),
    ('retention_time',      'continuous',  'plasma'),
    ('hydrophobicity',      'continuous',  'coolwarm'),
    ('spectrum_confidence', 'continuous',  'viridis'),
    ('modification_class',  'categorical', None),
    ('ptm_present',         'categorical', None),
    ('search_enzyme',       'categorical', None),
]
print('Hierarchy levels :', [c for c, _ in LEVELS])
print('Bottom properties:', [c for c, *_ in BOTTOM_PROPERTIES])

In [ ]:
# ── UMAP panel renderer + data-driven zoom helpers ────────────────────────────
def robust_bbox(d, pad=0.05):
    xs, ys = d[X].to_numpy(), d[Y].to_numpy()
    x0, x1 = np.percentile(xs, BBOX_PCT)
    y0, y1 = np.percentile(ys, BBOX_PCT)
    dx, dy = (x1 - x0) * pad, (y1 - y0) * pad
    return (x0 - dx, x1 + dx), (y0 - dy, y1 + dy)


def full_bbox(d, pad=0.04):
    """Whole-embedding framing. robust_bbox() trims the outer 1% (right for picking
    zoom windows) which cuts the detached blob at the top of the layout; use this
    whenever a figure is meant to show everything."""
    xs, ys = d[X].to_numpy(), d[Y].to_numpy()
    dx, dy = (xs.max() - xs.min()) * pad, (ys.max() - ys.min()) * pad
    return (xs.min() - dx, xs.max() + dx), (ys.min() - dy, ys.max() + dy)


def _topk(values):
    clean = [v for v in values if str(v).lower() not in _NULLISH]
    return [c for c, _ in Counter(clean).most_common(MAX_CATEGORIES)]


def region_of(bb):
    if bb is None:
        return df
    return df[(df[X] >= bb[0][0]) & (df[X] <= bb[0][1]) &
              (df[Y] >= bb[1][0]) & (df[Y] <= bb[1][1])]


def densest_subcluster_bbox(region):
    '''Spatially cluster a region and return the bbox of its densest sub-blob.'''
    xy = np.column_stack([region[X].to_numpy(), region[Y].to_numpy()])
    if len(xy) < 50:
        return robust_bbox(region, pad=0.08)
    km = KMeans(n_clusters=min(DESCENT_K, len(xy)), n_init=10, random_state=0).fit(xy)
    biggest = max(range(km.n_clusters), key=lambda c: int((km.labels_ == c).sum()))
    sub = region[km.labels_ == biggest]
    return robust_bbox(sub, pad=0.08)


# ── Fixed panel geometry ──────────────────────────────────────────────────────
# The UMAP box is always PANEL_IN inches square, whatever the legend contains.
# tight_layout() used to shrink the axes to make room for the legend, so a panel
# with 12 long category names ended up smaller than one with 3 — the embedding
# looked differently proportioned in every figure. Instead the axes is pinned to
# the whole base figure and the legend/colorbar are placed outside it; saving
# with bbox_inches='tight' grows the canvas around them rather than compressing
# the plot. Never call tight_layout() on these figures.
PANEL_IN = 5.4


def square_limits(xlim, ylim):
    """Equal data span on both axes, so the embedding is never stretched."""
    (x0, x1), (y0, y1) = xlim, ylim
    span   = max(x1 - x0, y1 - y0)
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    return (cx - span / 2, cx + span / 2), (cy - span / 2, cy + span / 2)


def new_umap_axes(size=PANEL_IN):
    fig = plt.figure(figsize=(size, size))
    ax  = fig.add_axes([0.0, 0.0, 1.0, 1.0])
    return fig, ax


def umap_legend(ax, **kw):
    """Legend outside the axes; it extends the canvas, it does not shrink the plot."""
    return ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
                     frameon=False, **kw)


def umap_colorbar(fig, sc, label):
    cax = fig.add_axes([1.03, 0.25, 0.028, 0.50])
    cb  = fig.colorbar(sc, cax=cax)
    cb.set_label(label, fontsize=10)
    cb.ax.tick_params(labelsize=8)
    return cb


PLOTLY_DIR = FIGURES_DIR / 'plotly'
PLOTLY_DIR.mkdir(parents=True, exist_ok=True)
_MPL_TO_PLOTLY = {'plasma': 'Plasma', 'coolwarm': 'RdBu_r', 'magma': 'Magma',
                  'viridis': 'Viridis', 'cividis': 'Cividis'}


def save_plotly_umap(name, d, color_col, kind, cmap, keep, cm, xlim, ylim):
    """Vector twin of a matplotlib UMAP panel, written to figures/3/plotly/.

    Same data, colours and framing; kept in its own directory so the curated set in
    figures/3 stays exactly what the paper uses. These are large (~19 MB at 100k
    points) because SVG stores every marker as an element.
    """
    import plotly.graph_objects as go
    fig = go.Figure()
    _x, _y = d[X].to_numpy(), d[Y].to_numpy()
    if kind == 'categorical':
        vals = d[color_col].astype(str).to_numpy()
        other = ~np.isin(vals, keep)
        if other.any():
            fig.add_trace(go.Scatter(x=_x[other], y=_y[other], mode='markers', name='Other',
                                     marker=dict(size=3, color=OTHER_GREY)))
        for cat in keep:
            m = vals == cat
            if m.any():
                fig.add_trace(go.Scatter(x=_x[m], y=_y[m], mode='markers', name=str(cat),
                                         marker=dict(size=3, color=cm.get(cat, OTHER_GREY))))
    else:
        c = pd.to_numeric(d[color_col], errors='coerce').to_numpy()
        fin = np.isfinite(c)
        fig.add_trace(go.Scatter(x=_x[fin], y=_y[fin], mode='markers', showlegend=False,
                                 marker=dict(size=3, color=c[fin], colorbar=dict(title=color_col),
                                             colorscale=_MPL_TO_PLOTLY.get(cmap, 'Viridis'))))
    fig.update_layout(template='simple_white', width=820, height=700,
                      margin=dict(l=60, r=40, t=20, b=60),
                      xaxis=dict(title='UMAP-1', range=list(xlim), showticklabels=False),
                      yaxis=dict(title='UMAP-2', range=list(ylim), showticklabels=False),
                      legend=dict(title=color_col))
    fig.write_image(str(PLOTLY_DIR / f'{name}.svg'))


def cat_cmap(col):
    """value -> colour for a column, straight from metadata_colors.json. Values
    with no key (and null-ish ones) are absent on purpose and fall to grey."""
    return META_COLORS.get(CAT_COLOR_BLOCK.get(col, ''), {})


def umap_panel(name, color_col, kind, xlim=None, ylim=None, exclude=(),
               cmap='magma', next_rect=None, figsize=(6.4, 5.4)):
    '''Render one UMAP panel. Points outside [xlim, ylim] are dropped (zoom).
    next_rect=(xlim, ylim) draws a dashed box marking where the next level zooms.'''
    set_publication_style()
    plt.rcParams.update({'axes.grid': False})

    d = df
    if exclude:
        d = d[~d[color_col].astype(str).isin([str(e) for e in exclude])]
    if xlim and ylim:
        d = d[(d[X] >= xlim[0]) & (d[X] <= xlim[1]) &
              (d[Y] >= ylim[0]) & (d[Y] <= ylim[1])]
    x, y = d[X].to_numpy(), d[Y].to_numpy()

    # Square the view so UMAP-1 and UMAP-2 share one scale. Frame with full_bbox,
    # not raw min/max: without its margin the extreme points sit exactly on the
    # axis and render clipped.
    _xl, _yl = square_limits(*((xlim, ylim) if (xlim and ylim) else full_bbox(d)))
    fig, ax = new_umap_axes()

    if kind == 'categorical':
        vals = d[color_col].astype(str).to_numpy()
        _cm  = cat_cmap(color_col)
        _seen = set(vals)
        keep = [c for c in _cm if c in _seen] or _topk(vals)
        mask_other = ~np.isin(vals, keep)
        ax.scatter(x[mask_other], y[mask_other], s=POINT_SIZE, c=OTHER_GREY,
                   alpha=0.4, linewidths=0, rasterized=True)
        for i, cat in enumerate(keep):
            m = vals == cat
            ax.scatter(x[m], y[m], s=POINT_SIZE,
                       color=_cm.get(cat, OTHER_GREY),
                       alpha=ALPHA, linewidths=0, rasterized=True, label=str(cat))
        umap_legend(ax, markerscale=3, fontsize=7, title=color_col)
    else:  # continuous
        cvals = pd.to_numeric(d[color_col], errors='coerce').to_numpy()
        finite = np.isfinite(cvals)
        sc = ax.scatter(x[finite], y[finite], s=POINT_SIZE, c=cvals[finite],
                        cmap=cmap, alpha=ALPHA, linewidths=0, rasterized=True)
        umap_colorbar(fig, sc, color_col)

    if next_rect is not None:
        (nx0, nx1), (ny0, ny1) = next_rect
        ax.add_patch(Rectangle((nx0, ny0), nx1 - nx0, ny1 - ny0, fill=False,
                     edgecolor='black', linewidth=1.6, linestyle='--', zorder=10))

    ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlim(*_xl); ax.set_ylim(*_yl)

    save_fig(fig, name)
    save_plotly_umap(name, d, color_col, kind, cmap,
                     keep if kind == 'categorical' else [],
                     _cm if kind == 'categorical' else {}, _xl, _yl)
    plt.show()


---
## Part 1 — Global view

The entire UMAP layout, recoloured by each property in turn.

In [ ]:
# ── Part 1 — whole-layout view, coloured by fragmentation method ─────────────
umap_panel('fig_1a_umap_frag_type', 'frag_type', 'categorical')

---
## Part 2 — Hierarchical zoom

Each level zooms into the densest sub-cluster of the level above and recolours by the next
field. The dashed black box marks where the next panel zooms in.

| Panel | Colour |
|---|---|
| level0 | fragmentation method (full space) |
| level1 | instrument |
| level2 | organism |
| level3 | precursor charge |

---
## Part 3 — Duplicate-spectrum retrieval

Spectra that share an identical peptide `sequence` are "duplicates"; good embeddings place
them as near neighbours. Each highlighted group is one peptide (distinct colour over grey).

In [ ]:
# ── Part 3 — duplicate-spectrum retrieval ─────────────────────────────────────
SEQ_COL = 'sequence'


def _dup_groups(top_k):
    '''Largest duplicate peptide groups (identical sequence, ≥2 spectra).'''
    # Deterministic order: count descending, then sequence ascending. pandas'
    # sort_values defaults to an unstable quicksort, so peptides with equal
    # group sizes ordered differently between numpy builds and changed which
    # ones fell inside top_k. sort_index first, then a stable value sort.
    vc = (df[df[SEQ_COL].notna()].groupby(SEQ_COL).size()
          .sort_index()
          .sort_values(ascending=False, kind='stable'))
    vc = vc[vc >= 2].head(top_k)
    return list(zip(vc.index.tolist(), vc.values.tolist()))


def _short(seq, n=22):
    s = str(seq)
    return s if len(s) <= n else s[:n - 1] + '…'


def duplicate_facet(name, top_k=12, ncols=4):
    '''Small multiples: one mini-panel per peptide over the shared grey layout.'''
    set_publication_style(); plt.rcParams.update({'axes.grid': False})
    groups = _dup_groups(top_k)
    x, y = df[X].to_numpy(), df[Y].to_numpy()
    seqs = df[SEQ_COL].astype(str).to_numpy()
    xlim, ylim = square_limits((x.min(), x.max()), (y.min(), y.max()))

    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 3.0 * nrows), squeeze=False)
    for j, ax in enumerate(axes.ravel()):
        if j >= len(groups):
            ax.axis('off'); continue
        seq, sz = groups[j]; m = seqs == str(seq)
        ax.scatter(x, y, s=2, c=OTHER_GREY, alpha=0.30, linewidths=0, rasterized=True)
        ax.scatter(x[m], y[m], s=12, color=CAT_COLORS[j % len(CAT_COLORS)], alpha=0.9,
                   linewidths=0.3, edgecolors='white', rasterized=True)
        ax.set_title(f'{_short(seq)}\n(n={sz})', fontsize=9)
        ax.set_xlim(*xlim); ax.set_ylim(*ylim)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ('top', 'right', 'bottom', 'left'):
            ax.spines[s].set_visible(True); ax.spines[s].set_linewidth(0.6)
    fig.tight_layout(); save_fig(fig, name); plt.show()


duplicate_facet('sup_fig_duplicate_retrieval_top12_facet', top_k=12)


---
## Part 4 — Composite main figure (paper layout)

The paper panel: one large **fragmentation-method** UMAP (2×2) with five smaller (1×1)
property-zoom insets arranged in an L. Each inset zooms into the region where that
property shows the **sharpest spatial separation** — continuous fields by largest local
gradient, categorical fields by the strongest boundary between two dominant categories.
Numbered boxes on the main panel mark where each inset zooms; inset borders match.

Tune `GRID_B` / `WIN` / `EXCLUDE_R` to change zoom resolution, or set
`REGION_OVERRIDES['<column>'] = ((x0, x1), (y0, y1))` to pin a region by hand.

In [ ]:
# ── Part 4 — composite config + property-driven zoom-region selection ─────────
import matplotlib.gridspec as gridspec

FRAG_COL = 'frag_type'
# (column, kind, cmap, label) — one inset each, in reading order around the main
# (column, kind, cmap, label, letter). Letters b-f are the categories that already
# had panels; peptide length and organism take i and j so the probe barplots at
# g/h keep the names they were given.
INSETS = [
    ('retention_time',    'continuous',  'plasma',   'Retention time',      'b'),
    ('instrument_family', 'categorical', None,       'Instrument',          'c'),
    ('enzyme_norm',       'categorical', None,       'Enzyme',              'd'),
    ('ptm_class',         'categorical', None,       'PTM class',           'e'),
    ('chem_label',        'categorical', None,       'Chemical labelling',  'f'),
    ('sequence_length',   'continuous',  'cividis',  'Peptide length',      'i'),
    ('organism_group',    'categorical', None,       'Organism',            'j'),
    ('detector_group',    'categorical', None,       'Detector',            'k'),
]
ZOOMS_PER_CAT = 3

# Region-selection params (each inset zooms where its property separates most)
GRID_B    = 22        # UMAP binning resolution for the search
WIN       = 3         # half-width of the zoom window, in bins. 2 cropped the
                      # structure, 4 pulled back too far; each window is then
                      # tightened onto its own points, so this sets magnification.
MIN_COUNT = 15        # ignore sparse bins
MIN_WIN_SPECTRA = 800 # a window must hold real data to deserve a panel
EXCLUDE_R = 3         # min bin separation between chosen inset centres
BOX_COLORS = ['#E4572E', '#17A398', '#7D3C98', '#2E86C1', '#B7950B']

# No manual pins. The enzyme panel used to be pinned to the whole right-hand
# cluster, which is exactly why its zoom stayed coarse; the separation criterion
# now picks its regions like every other category.
REGION_OVERRIDES = {}

_XE = np.linspace(df[X].min(), df[X].max(), GRID_B + 1)
_YE = np.linspace(df[Y].min(), df[Y].max(), GRID_B + 1)


def _bins(d):
    ix = np.clip(np.digitize(d[X].to_numpy(), _XE) - 1, 0, GRID_B - 1)
    iy = np.clip(np.digitize(d[Y].to_numpy(), _YE) - 1, 0, GRID_B - 1)
    return ix, iy


def _box_from_bin(bi, bj):
    i0, i1 = max(0, bi - WIN), min(GRID_B, bi + WIN + 1)
    j0, j1 = max(0, bj - WIN), min(GRID_B, bj + WIN + 1)
    return (_XE[i0], _XE[i1]), (_YE[j0], _YE[j1])


def score_continuous(col):
    # Score = local spatial gradient of the robust-normalised property
    p = pd.to_numeric(df[col], errors='coerce').to_numpy()
    fin = np.isfinite(p); d = df[fin]; pn = p[fin]
    lo, hi = np.nanpercentile(pn, [5, 95]); pn = np.clip((pn - lo) / (hi - lo + 1e-9), 0, 1)
    ix, iy = _bins(d)
    sums = np.zeros((GRID_B, GRID_B)); cnt = np.zeros((GRID_B, GRID_B))
    np.add.at(sums, (ix, iy), pn); np.add.at(cnt, (ix, iy), 1)
    valid = cnt >= MIN_COUNT
    mean_g = np.where(cnt > 0, sums / np.maximum(cnt, 1), np.nan)
    filled = np.where(valid, mean_g, np.nanmean(mean_g[valid]))
    gx, gy = np.gradient(filled)
    return np.sqrt(gx ** 2 + gy ** 2), valid


def score_categorical(col):
    # Score = strength of the sharpest boundary between two dominant categories
    vals = df[col].astype(str).to_numpy()
    keep = _topk(vals)
    cat = np.full(len(vals), -1)
    for i, c in enumerate(keep):
        cat[vals == c] = i
    mask = cat >= 0
    ix, iy = _bins(df); ix, iy, ci = ix[mask], iy[mask], cat[mask]
    counts = np.zeros((GRID_B, GRID_B, len(keep)))
    np.add.at(counts, (ix, iy, ci), 1)
    tot = counts.sum(2); dom = counts.argmax(2)
    purity = np.divide(counts.max(2), np.maximum(tot, 1))
    valid = tot >= MIN_COUNT
    score = np.zeros((GRID_B, GRID_B))
    for i in range(GRID_B):
        for j in range(GRID_B):
            if not valid[i, j]:
                continue
            best = 0.0
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                ni, nj = i + di, j + dj
                if 0 <= ni < GRID_B and 0 <= nj < GRID_B and valid[ni, nj] and dom[ni, nj] != dom[i, j]:
                    best = max(best, purity[i, j] * purity[ni, nj])
            score[i, j] = best
    return score, valid


CANDIDATES = 60       # windows scored before ranking; 3 are kept per category


def cluster_separation(win, col, kind, min_grp=40):
    """How far apart the groups inside a window sit, relative to their own spread.

    The old criterion scored the sharpness of a category boundary, which happily
    picked windows where two blobs merely touched. This scores the thing actually
    wanted: distance between group centroids divided by the groups' mean internal
    radius. A continuous property is cut into terciles first, so one rule covers
    both kinds. Higher = the clusters inside the zoom are further apart.
    """
    if kind == 'continuous':
        v = pd.to_numeric(win[col], errors='coerce')
        if v.notna().sum() < min_grp * 3:
            return -np.inf
        g = pd.qcut(v, 3, labels=False, duplicates='drop').to_numpy(dtype='float')
    else:
        g = pd.factorize(win[col].astype(str))[0].astype('float')

    xy = win[[X, Y]].to_numpy()
    cents, radii = [], []
    for _k in np.unique(g[np.isfinite(g)]):
        m = g == _k
        if m.sum() < min_grp:
            continue
        pts = xy[m]
        c   = pts.mean(0)
        cents.append(c)
        radii.append(float(np.linalg.norm(pts - c, axis=1).mean()))
    if len(cents) < 2:
        return -np.inf
    cents = np.asarray(cents)
    d = [np.linalg.norm(cents[a] - cents[b])
         for a in range(len(cents)) for b in range(a + 1, len(cents))]
    return float(np.mean(d) / max(1e-9, np.mean(radii)))


def pick_regions(col, kind, n=ZOOMS_PER_CAT):
    """The n best-separated, non-overlapping windows for one property.

    Scores every populated candidate with cluster_separation, ranks them, then
    walks the ranking taking windows whose centres are at least EXCLUDE_R bins
    apart so the three panels show genuinely different places. Each is framed on
    its own points rather than on the grid patch.
    """
    if col in REGION_OVERRIDES:
        return [(REGION_OVERRIDES[col], float('nan'))]
    score, valid = score_continuous(col) if kind == 'continuous' else score_categorical(col)
    s_ = score.copy(); s_[~valid] = -np.inf

    # Deterministic candidate order. np.argsort defaults to an unstable quicksort
    # and score_categorical produces many exactly-tied bins, so the order among
    # ties depended on the numpy build: 1.26 and 2.0 ranked them differently and
    # silently selected different zoom regions. lexsort gives a total order --
    # score descending, then bin indices ascending -- so ties resolve the same way
    # everywhere. Invalid (-inf) bins map to +inf and sort last, as before.
    cands = []
    _fi, _fj = np.unravel_index(np.arange(s_.size), s_.shape)
    for flat in np.lexsort((_fj, _fi, -s_.ravel())):
        if len(cands) >= CANDIDATES:
            break
        _i, _j = np.unravel_index(flat, s_.shape)
        if not np.isfinite(s_[_i, _j]):
            break
        _w = region_of(_box_from_bin(_i, _j))
        if len(_w) < MIN_WIN_SPECTRA:
            continue
        cands.append((cluster_separation(_w, col, kind), int(_i), int(_j)))

    # Tie-break on bin indices too: equal separations must order identically.
    cands.sort(key=lambda t: (-t[0], t[1], t[2]))
    taken = np.zeros(s_.shape, dtype=bool)
    out = []
    for sep, _i, _j in cands:
        if taken[_i, _j] or not np.isfinite(sep):
            continue
        taken[max(0, _i - EXCLUDE_R):_i + EXCLUDE_R + 1,
              max(0, _j - EXCLUDE_R):_j + EXCLUDE_R + 1] = True
        _w = region_of(_box_from_bin(_i, _j))
        out.append((robust_bbox(_w, pad=0.06), float(sep)))
        if len(out) == n:
            break
    return out


ZOOM_REGIONS = {}
for _col, _kind, _cmap, _lab, _letter in INSETS:
    ZOOM_REGIONS[_col] = pick_regions(_col, _kind)
    _txt = ', '.join(f'{len(region_of(_bb)):,} sp / sep {_s:.2f}'
                     for _bb, _s in ZOOM_REGIONS[_col])
    print(f'  {_lab:20s} {len(ZOOM_REGIONS[_col])} regions -> {_txt}')


---
### Part 4b — Standalone zoom panels

The same five property-zoom regions used in the composite above, each saved as its own
figure (no number badge / coloured border).

In [ ]:
# ── Part 4b — each composite inset region as a standalone zoom figure ─────────
def zoom_panel(name, col, kind, cmap, label, bb, exclude=()):
    set_publication_style(); plt.rcParams.update({'axes.grid': False})
    d = region_of(bb)
    if exclude:
        d = d[~d[col].astype(str).isin([str(e) for e in exclude])]
    x, y = d[X].to_numpy(), d[Y].to_numpy()
    fig, ax = new_umap_axes()
    if kind == 'continuous':
        c = pd.to_numeric(d[col], errors='coerce').to_numpy(); fin = np.isfinite(c)
        sc = ax.scatter(x[fin], y[fin], s=10, c=c[fin], cmap=cmap, alpha=0.85,
                        linewidths=0, rasterized=True)
        umap_colorbar(fig, sc, label)
    else:
        vals = d[col].astype(str).to_numpy()
        _cm  = cat_cmap(col)
        _seen = set(vals)
        keep = [c for c in _cm if c in _seen][:6] or _topk(vals)[:6]
        other = ~np.isin(vals, keep)
        ax.scatter(x[other], y[other], s=8, c=OTHER_GREY, alpha=0.4, linewidths=0, rasterized=True)
        for i, cat in enumerate(keep):
            m = vals == cat
            ax.scatter(x[m], y[m], s=11, color=_cm.get(cat, OTHER_GREY),
                       alpha=0.9, linewidths=0,
                       rasterized=True, label=str(cat))
        umap_legend(ax, markerscale=2, fontsize=8, title=label)
    _xl, _yl = square_limits(bb[0], bb[1])
    ax.set_xlim(*_xl); ax.set_ylim(*_yl); ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
    save_fig(fig, name)
    save_plotly_umap(name, d, col, kind, cmap,
                     keep if kind == 'categorical' else [],
                     _cm if kind == 'categorical' else {}, _xl, _yl)
    plt.show()


# Three regions per category, chosen in the config cell by cluster separation
for _col, _kind, _cmap, _lab, _letter in INSETS:
    for _k, (_bb, _sep) in enumerate(ZOOM_REGIONS[_col], start=1):
        zoom_panel(f'fig_1{_letter}_zoom_{_col}_{_k}', _col, _kind, _cmap, _lab, _bb)

# Enzymes also ship without trypsin, on the same three regions, so the pairs are
# directly comparable: recomputing regions on the subset would move the frames.
for _k, (_bb, _sep) in enumerate(ZOOM_REGIONS['enzyme_norm'], start=1):
    _n = int(region_of(_bb)['enzyme_norm'].astype(str).eq('Trypsin').sum())
    print(f'  fig_1d_zoom_enzyme_norm_{_k}_no_trypsin: {_n:,} tryptic spectra removed')
    zoom_panel(f'fig_1d_zoom_enzyme_norm_{_k}_no_trypsin', 'enzyme_norm', 'categorical',
               None, 'Enzyme', _bb, exclude=('Trypsin',))


---
### Part 4c — the same five properties over the whole embedding

Each zoom panel paired with its unzoomed counterpart, so the zoom can be read
against the layout it was cut from. Colours come from the shared per-column map,
so a category looks the same in both views.

In [ ]:
# ── Full-space twins of the five zoom panels ────────────────────────────────
# The enzyme panel ships in two versions. With trypsin it is the honest picture of
# the corpus but reads as one colour, since trypsin covers 93.8% of spectra; without
# it the alternative proteases become visible. Neither alone tells the whole story,
# so both are produced and the count removed is printed for the second.
UNZOOMED = [
    ('fig_1b_umap_retention_time',      'retention_time',    'continuous',  'plasma',  ()),
    ('fig_1c_umap_instruments',         'instrument_family', 'categorical', None,      ()),
    ('fig_1d_umap_enzymes',             'enzyme_norm',       'categorical', None,      ()),
    ('fig_1d_umap_enzymes_no_trypsin',  'enzyme_norm',       'categorical', None,      ('Trypsin',)),
    ('fig_1e_umap_ptms',                'ptm_class',         'categorical', None,      ()),
    ('fig_1f_umap_chemical_labelling',  'chem_label',        'categorical', None,      ()),
    ('fig_1i_umap_peptide_length',      'sequence_length',   'continuous',  'cividis', ()),
    ('fig_1j_umap_organisms',           'organism_group',    'categorical', None,      ()),
    ('fig_1k_umap_detectors',           'detector_group',    'categorical', None,      ()),
]

for _name, _col, _kind, _cmap, _excl in UNZOOMED:
    if _excl:
        _drop = df[_col].astype(str).isin([str(e) for e in _excl]).sum()
        print(f'  {_name}: excluding {list(_excl)} -> {int(_drop):,} of {len(df):,} '
              f'spectra removed ({_drop / len(df) * 100:.1f}%)')
    umap_panel(_name, _col, _kind, exclude=_excl,
               **({'cmap': _cmap} if _cmap else {}))

---
## Extra — organism clustering with human excluded

In [ ]:
# ── Extra — full-space UMAP coloured by organism, human excluded ──────────────
# Same whole-space view as Part 1, but every human-related spectrum (Homo sapiens,
# incl. co-assignments such as "Homo sapiens; Arabidopsis thaliana") is dropped so
# the non-human organism clusters become visible instead of being swamped.
set_publication_style()
plt.rcParams.update({'axes.grid': False})

_is_human  = df['search_organism'].astype(str).str.lower().str.contains('homo|human')
_nonhuman  = df[~_is_human]
print(f'Non-human spectra: {len(_nonhuman):,} of {len(df):,} '
      f'({len(_nonhuman) / len(df) * 100:.1f}%) across {_nonhuman["search_organism"].nunique()} organisms')

_x = _nonhuman[X].to_numpy()
_y = _nonhuman[Y].to_numpy()
_vals = _nonhuman['organism_group'].astype(str).to_numpy()
# hues come from metadata_colors.json, so an organism looks the same everywhere
_cm      = cat_cmap('organism_group')
_present = set(_vals)
_keep    = [o for o in _cm if o in _present]
_mask_other = ~np.isin(_vals, _keep)

fig, ax = new_umap_axes()
# Keep the full-embedding framing so positions are comparable to the other panels
(_xlim, _ylim) = full_bbox(df)

ax.scatter(_x[_mask_other], _y[_mask_other], s=POINT_SIZE, c=OTHER_GREY,
           alpha=0.4, linewidths=0, rasterized=True, label='Other')
for i, cat in enumerate(_keep):
    m = _vals == cat
    ax.scatter(_x[m], _y[m], s=POINT_SIZE,
               color=_cm.get(cat, OTHER_GREY),
               alpha=ALPHA, linewidths=0, rasterized=True, label=str(cat))

umap_legend(ax, markerscale=3, fontsize=7, title='Organism')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.set_xticks([]); ax.set_yticks([])
_xlim, _ylim = square_limits(_xlim, _ylim)
ax.set_xlim(*_xlim); ax.set_ylim(*_ylim)

save_fig(fig, 'sup_fig_full_13_umap_all_no_human')
save_plotly_umap('sup_fig_full_13_umap_all_no_human', _nonhuman, 'organism_group',
                 'categorical', None, _keep, _cm, _xlim, _ylim)
plt.show()


---
# Part 5 — Do organisms cluster by biology, or by project (batch)?

A cluster of one organism is only scientifically meaningful if it is **not merely
a project (batch) cluster in disguise**. If every organism came from its own
project, organism-clustering would be trivial. The interesting case is the one
this part tests: an organism whose spectra span **several projects** that
nonetheless **intermix** in the embedding — that is genuine biological
separation, not batch memorisation.

**Scope caveat.** Organism and project are confounded here (Cramér's V ≈ 0.63),
and **~83 % of organisms come from a single project**, so they cannot be tested
at all. Only organisms spanning ≥ 2 projects (and ≥ 300 spectra) are testable.
The dataset is also **~75 % Homo sapiens**, so a *global* "cluster vs label"
correlation is misleading (organism can't define clusters when one label owns
3/4 of the data); the honest test is done **within each organism**.

**Method.** For each testable organism we measure the *project silhouette* on its
UMAP points: **< 0 → projects mixed (biology)**, **> 0 → projects separated
(batch)**. We also compute the *instrument silhouette* as a baseline — if project
separation merely tracks the instrument, it is legitimate technical structure
(the UMAP is known to organise by fragmentation→instrument), not a spurious
batch effect.

1. **Diagnostic** — project vs instrument silhouette per organism.
2. **Visual check** — each organism's UMAP region, coloured by project.


In [ ]:
# ── Part 5a — biology-vs-batch diagnostic (project vs instrument silhouette) ──
from sklearn.metrics import silhouette_score
from scipy.stats import chi2_contingency
import matplotlib.patches as mpatches
set_publication_style()

ORG, PROJ, INST, FRAG = 'search_organism', 'search_project', 'search_instrument', 'search_fragmentation'
MIN_SPEC = 300   # organisms below this are too small for a reliable silhouette

# local frame: drop missing + mixed-organism labels (e.g. "Homo sapiens; Arabidopsis")
_dfb = df.dropna(subset=[ORG, PROJ]).copy()
_dfb = _dfb[~_dfb[ORG].astype(str).str.contains(';')]


def _cramers_v(a, b):
    ct = pd.crosstab(a, b); chi2 = chi2_contingency(ct)[0]; n = ct.values.sum(); r, k = ct.shape
    phi2c = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2c / max(1e-12, min(kc - 1, rc - 1)))


def _safe_sil(coords, labels, cap=4000, seed=0):
    labels = np.asarray(labels)
    if pd.Series(labels).nunique() < 2:
        return np.nan
    if len(labels) > cap:
        idx = np.random.RandomState(seed).choice(len(labels), cap, replace=False)
        coords, labels = coords[idx], labels[idx]
    try:
        return float(silhouette_score(coords, labels))
    except Exception:
        return np.nan


_g = _dfb.groupby(ORG)[PROJ].agg(n_proj='nunique', n_spec='count')
_testable = _g[(_g.n_proj >= 2) & (_g.n_spec >= MIN_SPEC)].sort_values('n_spec', ascending=False)
_rows = []
for o in _testable.index:
    sub = _dfb[_dfb[ORG] == o]; co = sub[[X, Y]].values
    _rows.append(dict(organism=o, n_proj=int(sub[PROJ].nunique()), n_spec=len(sub),
                      sil_project=_safe_sil(co, sub[PROJ]),
                      sil_instrument=_safe_sil(co, sub[INST]),
                      sil_frag=_safe_sil(co, sub[FRAG])))
diag = pd.DataFrame(_rows)


def _verdict(r):
    if pd.isna(r.sil_project):                                   return 'n/a'
    if r.sil_project <= 0.10:                                    return 'biology'
    if pd.isna(r.sil_instrument):                                return 'batch'
    if r.sil_project - r.sil_instrument <= 0.10:                 return 'technical'
    return 'batch'


diag['verdict'] = diag.apply(_verdict, axis=1)
_V = _cramers_v(_dfb[ORG], _dfb[PROJ])
print(f'Global Cramer V(organism, project) = {_V:.3f}   (0 = independent, 1 = fully confounded)')
print(diag[['organism', 'n_proj', 'n_spec', 'sil_project', 'sil_instrument', 'verdict']].to_string(index=False))


In [ ]:
# ── Homo sapiens alone, coloured by project ──────────────────────────────────
# H. sapiens carries 46 of the 57 projects, so inside the shared 4-panel figure it
# can only ever be partly coloured. Given the full colour budget to itself its
# coverage rises substantially, and the batch structure becomes readable.
set_publication_style()
plt.rcParams.update({'axes.grid': False})

_hs   = _dfb[_dfb[ORG] == 'Homo sapiens']
# Project colours come from metadata_colors.json like everything else. Projects
# without a key there fall to grey rather than borrowing another project's hue.
_cmap = cat_cmap(PROJ)
_keep = [p for p in _cmap if p in set(_hs[PROJ].astype(str))]
_cov  = _hs[PROJ].isin(_keep).mean() * 100

fig, ax = new_umap_axes(6.2)
ax.scatter(df[X], df[Y], s=2, c='#EEEEEE', alpha=0.5, linewidths=0, rasterized=True)
_om = (~_hs[PROJ].isin(_keep)).values
if _om.any():
    ax.scatter(_hs[X].values[_om], _hs[Y].values[_om], s=4, c=OTHER_GREY,
               alpha=0.6, linewidths=0, rasterized=True, label='other')
for _p in _keep:
    _m = (_hs[PROJ] == _p).values
    ax.scatter(_hs[X].values[_m], _hs[Y].values[_m], s=4, color=_cmap[_p],
               alpha=0.8, linewidths=0, rasterized=True, label=str(_p))

_xl, _yl = square_limits(*full_bbox(df))
ax.set_xlim(*_xl); ax.set_ylim(*_yl)
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
umap_legend(ax, markerscale=4, fontsize=7.5, title='Project')
save_fig(fig, 'sup_fig_organism_vs_project_human')
save_plotly_umap('sup_fig_organism_vs_project_human', _hs, PROJ,
                 'categorical', None, _keep, _cmap, _xl, _yl)
plt.show()

print(f'Homo sapiens: {len(_hs):,} spectra across {_hs[PROJ].nunique()} projects; '
      f'top {len(_keep)} cover {_cov:.1f}%')

---
## Part 6 — same peptide with vs. without PTM

Does a peptide carrying a PTM (e.g. `M[UNIMOD:35]`, oxidation) land in the **same UMAP region** as its unmodified twin? We pair spectra by their `unmodified_peptide` backbone, keep the backbones that appear both modified and unmodified, and measure the UMAP distance between the two centroids — benchmarked against a random-pair baseline and the intra-replicate floor (two spectra of the *same* unmodified peptide).

---
# Part 7 — Is the embedding biology, or something else?

Three complementary tests. **7a** asks which metadata field the embedding actually
tracks. **7b** asks whether those fields are even separable from one another.
**7c** is the causal test: does the *same peptide* land in the same place when it
comes from a different project and a different instrument?

In [ ]:
# ── Part 7a — which field does the embedding actually track? ─────────────────
# For every spectrum, look at its k nearest neighbours in the embedding and ask
# what fraction share its label. A field the embedding encodes gives high purity.
# Raw purity is not comparable across fields — a field with 2 huge categories
# scores high by chance — so each is reported against its own shuffled baseline:
#     enrichment = (purity - chance) / (1 - chance)
# 0 = the embedding knows nothing about this field, 1 = it separates it perfectly.
from sklearn.neighbors import NearestNeighbors

set_publication_style()
K_NN     = 50
SHUFFLES = 3

_coords = df[[X, Y]].to_numpy()
_nn     = NearestNeighbors(n_neighbors=K_NN + 1).fit(_coords)
_idx    = _nn.kneighbors(_coords, return_distance=False)[:, 1:]   # drop self

def _purity(labels):
    lab = pd.factorize(pd.Series(labels).astype(str))[0]
    return float((lab[_idx] == lab[:, None]).mean())

def _enrichment(labels, seed=0):
    obs  = _purity(labels)
    rng  = np.random.RandomState(seed)
    lab  = np.asarray(labels)
    base = np.mean([_purity(rng.permutation(lab)) for _ in range(SHUFFLES)])
    return obs, base, (obs - base) / max(1e-9, 1 - base)

# sequence_length is continuous — bin it so it can be scored the same way
_seq_bin = pd.qcut(df['sequence_length'], 5, duplicates='drop').astype(str)

FIELDS = [
    ('search_organism',   'Organism',            'biology'),
    ('precursor_charge',  'Precursor charge',    'biology'),
    ('modification_class','Modification class',  'biology'),
    ('ptm_present',       'PTM present',         'biology'),
    ('__seqlen',          'Peptide length',      'biology'),
    ('search_project',    'Project',             'technical'),
    ('search_instrument', 'Instrument',          'technical'),
    ('search_detector',   'Detector',            'technical'),
    ('frag_type',         'Fragmentation',       'technical'),
    ('search_acquisition','Acquisition',         'technical'),
    ('search_enzyme',     'Enzyme (protocol)',   'technical'),
]

_rows = []
for col, label, group in FIELDS:
    vals = _seq_bin if col == '__seqlen' else df[col]
    if col != '__seqlen' and col not in df.columns:
        continue
    obs, base, enr = _enrichment(vals.to_numpy() if hasattr(vals, 'to_numpy') else vals)
    _rows.append(dict(field=label, group=group, purity=obs, chance=base,
                      enrichment=enr, n_cat=pd.Series(vals).nunique()))
knn_diag = pd.DataFrame(_rows).sort_values('enrichment')
print(knn_diag.to_string(index=False))

_GCOL = {'biology': '#2E75B6', 'technical': '#D62728'}
_ys   = np.arange(len(knn_diag))

fig, ax = plt.subplots(figsize=(9.6, 5.2))
ax.grid(axis='y', visible=False)
for _y, (_, r) in zip(_ys, knn_diag.iterrows()):
    ax.plot([0, r.enrichment], [_y, _y], color=_GCOL[r.group], lw=2.2, zorder=2)
    ax.scatter(r.enrichment, _y, s=130, color=_GCOL[r.group], zorder=4,
               edgecolor='white', lw=1.0)
    ax.text(r.enrichment + 0.018, _y, f'{r.enrichment:.2f}', va='center',
            fontsize=8.5, color='#555')
ax.set_yticks(_ys); ax.set_yticklabels(knn_diag['field'])
ax.set_xlim(0, max(0.35, knn_diag.enrichment.max() * 1.22))
ax.set_xlabel(f'{K_NN}-NN label enrichment over chance', fontsize=12)
ax.text(0.0, -0.155, '0 = the embedding ignores this field   ·   1 = it separates it perfectly',
        transform=ax.transAxes, fontsize=9, color='#888', va='top')
ax.legend(handles=[mpatches.Patch(color=_GCOL['biology'],   label='biological / analyte'),
                   mpatches.Patch(color=_GCOL['technical'], label='technical / acquisition')],
          loc='lower right', fontsize=9)
fig.tight_layout()
save_fig(fig, 'sup_fig_diag_knn_field_enrichment')
plt.show()

In [ ]:
# ── Part 7b — are those fields even separable? ───────────────────────────────
# 7a is only interpretable if the fields are independent. If every organism comes
# from one project on one instrument, "organism" and "project" are the same column
# wearing different names and no ranking can tell them apart. Cramer V: 0 =
# independent, 1 = one field fully determines the other.
set_publication_style()

_CONF_FIELDS = [(c, l) for c, l, _ in FIELDS if c != '__seqlen']
_labels      = [l for _, l in _CONF_FIELDS]
_n           = len(_CONF_FIELDS)

_V = np.zeros((_n, _n))
for _i, (_ci, _) in enumerate(_CONF_FIELDS):
    for _j, (_cj, _) in enumerate(_CONF_FIELDS):
        _V[_i, _j] = 1.0 if _i == _j else _cramers_v(df[_ci].astype(str), df[_cj].astype(str))

fig, ax = plt.subplots(figsize=(7.6, 6.4))
ax.grid(False)
_im = ax.imshow(_V, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(_n)); ax.set_xticklabels(_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(_n)); ax.set_yticklabels(_labels, fontsize=9)
for _i in range(_n):
    for _j in range(_n):
        ax.text(_j, _i, f'{_V[_i, _j]:.2f}', ha='center', va='center', fontsize=7.5,
                color='white' if _V[_i, _j] > 0.55 else '#333')
fig.colorbar(_im, ax=ax, shrink=0.8, label="Cramer V")
fig.tight_layout()
save_fig(fig, 'sup_fig_diag_metadata_confounding')
plt.show()

_off = _V[np.triu_indices(_n, k=1)]
print(f'strongest confounding pairs (Cramer V >= 0.6):')
for _i in range(_n):
    for _j in range(_i + 1, _n):
        if _V[_i, _j] >= 0.6:
            print(f'  {_labels[_i]:20s} <-> {_labels[_j]:20s} {_V[_i, _j]:.2f}')

In [ ]:
# ── Part 7c — the causal test: same peptide, different batch ─────────────────
# If the embedding encodes the analyte, two spectra of the SAME peptide should sit
# close together even when they come from different projects and instruments. If
# it encodes the batch, they should separate, and unrelated peptides from the same
# project should instead be close. Four pair populations, distances normalised so
# 1.0 = the median distance between two spectra picked at random.
set_publication_style()
PAIRS_PER_GROUP = 40000
_rng = np.random.RandomState(0)

_xy   = df[[X, Y]].to_numpy()
_seq  = df[SEQ_COL].astype(str).to_numpy()
_proj = df[PROJ].astype(str).to_numpy()
_inst = df[INST].astype(str).to_numpy()

def _d(i, j):
    return np.linalg.norm(_xy[i] - _xy[j], axis=1)

# random baseline sets the unit
_ra, _rb = (_rng.randint(0, len(df), PAIRS_PER_GROUP),
            _rng.randint(0, len(df), PAIRS_PER_GROUP))
_keep    = _ra != _rb
_rand_d  = _d(_ra[_keep], _rb[_keep])
_UNIT    = float(np.median(_rand_d))

# index spectra by peptide, keep peptides seen at least twice
_by_pep = {}
for _i, _s in enumerate(_seq):
    _by_pep.setdefault(_s, []).append(_i)
_dups = {k: np.asarray(v) for k, v in _by_pep.items() if len(v) >= 2}
print(f'{len(_dups):,} peptides with >= 2 spectra '
      f'({sum(len(v) for v in _dups.values()):,} spectra)')

_same_pep_same_proj, _same_pep_diff_proj, _same_pep_diff_inst = [], [], []
_pep_keys = list(_dups)
for _ in range(PAIRS_PER_GROUP):
    _k = _pep_keys[_rng.randint(len(_pep_keys))]
    _g = _dups[_k]
    _i, _j = _rng.choice(_g, 2, replace=False)
    _dist = float(np.linalg.norm(_xy[_i] - _xy[_j]))
    (_same_pep_same_proj if _proj[_i] == _proj[_j] else _same_pep_diff_proj).append(_dist)
    if _inst[_i] != _inst[_j]:
        _same_pep_diff_inst.append(_dist)

# control: different peptides drawn from the same project
_diff_pep_same_proj = []
_proj_index = {}
for _i, _p in enumerate(_proj):
    _proj_index.setdefault(_p, []).append(_i)
_proj_index = {k: np.asarray(v) for k, v in _proj_index.items() if len(v) >= 2}
_pkeys = list(_proj_index)
while len(_diff_pep_same_proj) < PAIRS_PER_GROUP:
    _g = _proj_index[_pkeys[_rng.randint(len(_pkeys))]]
    _i, _j = _rng.choice(_g, 2, replace=False)
    if _seq[_i] != _seq[_j]:
        _diff_pep_same_proj.append(float(np.linalg.norm(_xy[_i] - _xy[_j])))

POPS = [
    ('same peptide · same project',      np.asarray(_same_pep_same_proj), '#1B5E20'),
    ('same peptide · different project', np.asarray(_same_pep_diff_proj), '#2E75B6'),
    ('same peptide · different instrument', np.asarray(_same_pep_diff_inst), '#7FB3D3'),
    ('different peptide · same project', np.asarray(_diff_pep_same_proj), '#D62728'),
    ('random pair',                      _rand_d,                         '#999999'),
]

fig, ax = plt.subplots(figsize=(9.6, 5.4))
ax.grid(axis='x', visible=False)
for _lab, _v, _c in POPS:
    if len(_v) < 20:
        continue
    _s = np.sort(_v / _UNIT)
    ax.plot(_s, np.arange(1, len(_s) + 1) / len(_s), color=_c, lw=2.2,
            label=f'{_lab}  (median {np.median(_s):.2f}, n={len(_s):,})')
ax.axvline(1.0, color='#444', lw=1.0, ls='--')
ax.set_xscale('log', base=10)
ax.set_xlabel('UMAP distance   (1.0 = median distance between two random spectra)')
ax.set_ylabel('Cumulative fraction of pairs')
ax.legend(loc='upper left', fontsize=8.5)
fig.tight_layout()
save_fig(fig, 'sup_fig_diag_same_peptide_across_batches')
plt.show()

print('\nmedian normalised distance')
for _lab, _v, _ in POPS:
    if len(_v) >= 20:
        print(f'  {_lab:38s} {np.median(_v) / _UNIT:5.2f}   (n={len(_v):,})')

---
# Part 8 — the top-12 duplicate peptides, recoloured by batch metadata

Same twelve peptides and same layout as `duplicate_retrieval_top12_facet`, but each
peptide's spectra are coloured by where they came from. If one peptide's spectra
split into separate blobs that track the colour, the embedding is placing that
peptide by its run rather than by its identity.

In [ ]:
# ── Top-12 duplicate peptides, coloured by acquisition metadata ──────────────
# One mini-panel per peptide over the shared grey embedding, exactly as in
# duplicate_retrieval_top12_facet, but coloured by project / instrument /
# fragmentation instead of one flat colour per peptide.
set_publication_style()
plt.rcParams.update({'axes.grid': False})

FACET_TOP_K = 12
FACET_NCOLS = 4


def duplicate_facet_by(name, color_col, nice, top_k=FACET_TOP_K, ncols=FACET_NCOLS):
    groups = _dup_groups(top_k)
    _seqs  = df[SEQ_COL].astype(str).to_numpy()
    _vals  = df[color_col].astype(str).to_numpy()
    _x, _y = df[X].to_numpy(), df[Y].to_numpy()
    _member = np.isin(_seqs, [str(sq) for sq, _ in groups])

    # One colour map shared by every panel: a category keeps its colour throughout,
    # so a hue means the same run in panel 1 and in panel 12.
    _keep = [c for c, _ in Counter(_vals[_member]).most_common(len(CAT_COLORS))]
    _cmap = {c: CAT_COLORS[i % len(CAT_COLORS)] for i, c in enumerate(_keep)}

    _xlim, _ylim = square_limits((_x.min(), _x.max()), (_y.min(), _y.max()))
    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 3.15 * nrows),
                             squeeze=False)

    _summary = []
    for j, ax in enumerate(axes.ravel()):
        if j >= len(groups):
            ax.axis('off'); continue
        seq, sz = groups[j]
        m = _seqs == str(seq)
        ax.scatter(_x, _y, s=2, c='#EEEEEE', alpha=0.5, linewidths=0, rasterized=True)
        sub = _vals[m]
        om  = ~np.isin(sub, _keep)
        if om.any():
            ax.scatter(_x[m][om], _y[m][om], s=13, c=OTHER_GREY, alpha=0.8,
                       linewidths=0.3, edgecolors='white', rasterized=True)
        for cat in _keep:
            cm = sub == cat
            if cm.any():
                ax.scatter(_x[m][cm], _y[m][cm], s=13, color=_cmap[cat], alpha=0.9,
                           linewidths=0.3, edgecolors='white', rasterized=True)
        _ncat = int(pd.Series(sub).nunique())
        _summary.append((str(seq), sz, _ncat))
        ax.set_title(f'{_short(seq)}\n n={sz} · {_ncat} {nice.lower()}'
                     + ('s' if _ncat != 1 else ''), fontsize=8.5)
        ax.set_xlim(*_xlim); ax.set_ylim(*_ylim)
        ax.set_xticks([]); ax.set_yticks([])

    handles = [plt.Line2D([0], [0], marker='o', linestyle='', markersize=6,
                          markerfacecolor=_cmap[c], markeredgecolor='white', label=c)
               for c in _keep]
    fig.legend(handles=handles, loc='lower center', ncol=min(6, len(handles)),
               frameon=False, fontsize=8, title=nice,
               bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=[0, 0.03, 1, 1])
    save_fig(fig, name)
    plt.show()
    return _summary


_FACETS = [
    ('sup_fig_duplicate_top12_by_project',       PROJ,        'Project'),
    ('sup_fig_duplicate_top12_by_instrument',    INST,        'Instrument'),
    ('sup_fig_duplicate_top12_by_fragmentation', 'frag_type', 'Fragmentation method'),
]

_spread = {}
for _name, _col, _nice in _FACETS:
    _spread[_nice] = duplicate_facet_by(_name, _col, _nice)

# how far each of the twelve peptides is spread across batches
_tab = pd.DataFrame({'peptide': [s for s, _, _ in _spread['Project']],
                     'spectra': [n for _, n, _ in _spread['Project']]})
for _nice in _spread:
    _tab[_nice] = [c for _, _, c in _spread[_nice]]
print('\nHow many distinct batches each top-12 peptide spans:')
print(_tab.to_string(index=False))

---
## Part 9 — linear-probe performance on the embedding

How well a linear probe recovers each property from the embedding: R² for the
continuous targets, macro-F1 for the categorical ones.

In [ ]:
# ── Linear-probe results ─────────────────────────────────────────────────────
# NOT derived from umap_coordinates.parquet: these are probe scores produced
# outside this notebook, so they are declared here as constants. If a results
# file appears, read them from it instead of editing these literals.
set_publication_style()
plt.rcParams.update({'axes.grid': False})

PROBE_R2 = [
    ('Spectrum confidence', 0.973),
    ('Precursor m/z',       0.929),
    ('Precursor mass',      0.732),
    ('Hydrophobicity',      0.605),
]
PROBE_F1 = [
    ('Fragmentation method', 0.692),
    ('Instrument (13-class)', 0.807),
    ('Precursor charge',     0.583),
    ('PTM presence',         0.777),
]

BAR_FILL = META_COLORS['pastel'][0]      # pastel blue, from the shared colour map


def probe_bars(name, panel, rows, ylabel):
    fig, ax = plt.subplots(figsize=get_figsize(1))
    _labels = [l for l, _ in rows]
    _vals   = [v for _, v in rows]
    _x      = np.arange(len(rows))

    ax.bar(_x, _vals, width=0.62, color=BAR_FILL,
           edgecolor='black', linewidth=0.8, zorder=3)
    for _xi, _v in zip(_x, _vals):
        ax.text(_xi, _v + 0.018, f'{_v:.3f}', ha='center', va='bottom', fontsize=10)

    ax.set_ylabel(ylabel)
    ax.set_ylim(0.0, 1.0)
    ax.set_yticks(np.arange(0.0, 1.01, 0.2))
    ax.set_xticks(_x)
    ax.set_xticklabels(_labels, rotation=45, ha='right')
    ax.set_xlim(-0.6, len(rows) - 0.4)
    # panel letter, outside the axes so it never collides with a bar
    ax.text(-0.16, 1.04, panel, transform=ax.transAxes, ha='left', va='bottom',
            fontsize=15, fontweight='bold')
    sns.despine(ax=ax)
    fig.tight_layout()
    save_fig(fig, name)
    plt.show()


probe_bars('fig_1g_probe_regression',     'B', PROBE_R2, r'R$^2$ (regression)')
probe_bars('fig_1h_probe_classification', 'C', PROBE_F1, 'Macro-F1 (classification)')